# Student Performance & Placement Predictor

**Goal:** Analyze student academic and extracurricular data to understand which
factors are associated with campus placement, and build machine learning models
that predict whether a student is likely to be placed.

**Dataset:** This project uses a **synthetically generated dataset** (created in
`src/generate_dataset.py`) designed to mimic realistic relationships between
student attributes and placement outcomes. It is NOT real student data collected
from any institution. Results demonstrate the analysis *workflow* correctly, but
should not be read as real-world findings about what determines student placement.

**Author:** _[Your Name Here]_


## 1. Import Libraries

In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report
)

from data_preprocessing import (
    load_data, clean_data, engineer_features,
    get_feature_columns, build_preprocessor
)

pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")
RANDOM_STATE = 42


## 2. Load Dataset

In [ ]:
df = load_data("../data/student_placement_data.csv")
df.head()


## 3. Data Understanding

Before touching the data, we check its shape, types, and quality. This tells
us what cleaning is actually necessary — we don't guess, we check.

In [ ]:
print("Rows, Columns:", df.shape)


In [ ]:
df.head()


In [ ]:
df.tail()


In [ ]:
df.columns


In [ ]:
df.dtypes


In [ ]:
df.info()


In [ ]:
df.describe()


**What `.describe()` tells us:** count, mean, std, min, max, and quartiles
for every numeric column. This is how we'd spot impossible values (e.g.
attendance > 100%) early. Here, CGPA maxes at ~10 and attendance at 100 —
no impossible values found.

In [ ]:
df.isnull().sum()


In [ ]:
print("Duplicate rows:", df.duplicated().sum())


In [ ]:
df['Placement_Status'].value_counts()


In [ ]:
df['Placement_Status'].value_counts(normalize=True) * 100


**Takeaway:** we have missing values in 4 columns and 15 duplicate rows —
both need to be addressed. The target is close to balanced (~51% Not Placed,
49% Placed), so we won't need special imbalance-handling techniques, but we
will still use `stratify` in our train-test split as good practice.

## 4. Data Cleaning

**Decisions and reasoning:**
- **Duplicate rows → dropped.** A duplicate means the same student record
  appears twice, which would let the model "see" that student's outcome
  more than once and bias training.
- **Missing numeric values (CGPA, Attendance_Percentage, Communication_Score)
  → filled with the column median.** Median is robust to skew, and with only
  ~2-3% missing in each column, imputing preserves more data than dropping rows.
- **Missing categorical values (Extracurricular_Activities) → filled with the
  column mode.** Standard simple approach for a small fraction of missing data.
- We do **not** blindly drop rows with any missing value, since that would
  throw away otherwise-good data unnecessarily.

In [ ]:
df_clean = clean_data(df)
print("Shape after cleaning:", df_clean.shape)
print("Missing values remaining:", df_clean.isnull().sum().sum())
print("Duplicates remaining:", df_clean.duplicated().sum())


## 5. Exploratory Data Analysis (EDA)

We now visualize relationships between features and placement. For each
chart we note what we're plotting, why, and what it tells us — being careful
not to claim causation just because two variables are correlated.

### 5.1 Placement Status Distribution

In [ ]:
sns.countplot(data=df_clean, x="Placement_Status", hue="Placement_Status", palette="Set2", legend=False)
plt.title("Placement Status Distribution")
plt.show()


**What it tells us:** classes are close to balanced, so accuracy is a reasonably meaningful metric here (though we'll still look at F1-score and ROC-AUC).

### 5.2 CGPA and Attendance Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df_clean["CGPA"], kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("CGPA Distribution")
sns.histplot(df_clean["Attendance_Percentage"], kde=True, ax=axes[1], color="darkorange")
axes[1].set_title("Attendance Percentage Distribution")
plt.tight_layout()
plt.show()


**What it tells us:** both are roughly bell-shaped (normal-like), with no extreme outliers — consistent with realistic student populations.

### 5.3 Study Hours Distribution

In [ ]:
sns.histplot(df_clean["Study_Hours_Per_Day"], kde=True, color="seagreen")
plt.title("Study Hours Per Day Distribution")
plt.show()


### 5.4 CGPA vs Placement Status

In [ ]:
sns.boxplot(data=df_clean, x="Placement_Status", y="CGPA", hue="Placement_Status", palette="Set2", legend=False)
plt.title("CGPA vs Placement Status")
plt.show()


**What it tells us:** Placed students show a visibly higher median CGPA than Not Placed students. This is the strongest individual numeric relationship with placement in this dataset — though as we'll see in the correlation heatmap, it's still only a moderate relationship, not a strong deterministic one.

### 5.5 Attendance vs Placement Status

In [ ]:
sns.boxplot(data=df_clean, x="Placement_Status", y="Attendance_Percentage", hue="Placement_Status", palette="Set2", legend=False)
plt.title("Attendance vs Placement Status")
plt.show()


**What it tells us:** the difference between groups is much smaller here than for CGPA — attendance alone doesn't separate Placed from Not Placed very clearly.

### 5.6 Internships and Projects vs Placement Status

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df_clean, x="Internships", hue="Placement_Status", palette="Set2", ax=axes[0])
axes[0].set_title("Internships vs Placement Status")
sns.countplot(data=df_clean, x="Projects", hue="Placement_Status", palette="Set2", ax=axes[1])
axes[1].set_title("Projects vs Placement Status")
plt.tight_layout()
plt.show()


**What it tells us:** students with more internships/projects tend to have a slightly higher share of Placed outcomes, though the effect is gradual rather than a sharp cutoff.

### 5.7 Technical Skill and Communication Score vs Placement

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df_clean, x="Placement_Status", y="Technical_Skill_Score", hue="Placement_Status", palette="Set2", legend=False, ax=axes[0])
axes[0].set_title("Technical Skill Score vs Placement")
sns.boxplot(data=df_clean, x="Placement_Status", y="Communication_Score", hue="Placement_Status", palette="Set2", legend=False, ax=axes[1])
axes[1].set_title("Communication Score vs Placement")
plt.tight_layout()
plt.show()


### 5.8 Backlogs vs Placement Status

In [ ]:
sns.countplot(data=df_clean, x="Backlogs", hue="Placement_Status", palette="Set2")
plt.title("Backlogs vs Placement Status")
plt.show()


**What it tells us:** students with more backlogs show a somewhat lower share of Placed outcomes — a mild negative relationship, matching intuition.

### 5.9 Correlation Heatmap

In [ ]:
df_fe_temp = engineer_features(df_clean)
numeric_cols = [
    "Age", "CGPA", "Attendance_Percentage", "Study_Hours_Per_Day", "Backlogs",
    "Internships", "Projects", "Certifications", "Communication_Score",
    "Technical_Skill_Score", "Aptitude_Score", "Total_Experience",
    "Skill_Average", "Project_Experience"
]
df_corr = df_fe_temp.copy()
df_corr["Placement_Status_Num"] = (df_corr["Placement_Status"] == "Placed").astype(int)
corr = df_corr[numeric_cols + ["Placement_Status_Num"]].corr()

plt.figure(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap (Numeric Features + Placement)")
plt.show()


In [ ]:
corr["Placement_Status_Num"].drop("Placement_Status_Num").sort_values(ascending=False)


**What it tells us:** no single numeric feature shows a strong (>0.3)
linear correlation with placement — CGPA (~0.18) is the strongest, followed
by Internships and Project_Experience. This suggests placement here results
from a *combination* of many weak-to-moderate factors (plus randomness),
not one dominant variable. Note: `Total_Experience` and `Project_Experience`
correlate strongly with the raw columns they're built from (e.g. Internships)
— expected, since they're derived features, not independent new information.

**Important:** correlation is not causation. A higher CGPA being associated
with placement doesn't prove CGPA *causes* placement — other unmeasured
factors likely influence both.

## 6. Feature Engineering

We create a few combined features that may carry useful signal:

| Feature | Formula | Why it might help |
|---|---|---|
| `Total_Experience` | Internships + (had previous internship) | Combines two "prior experience" signals |
| `Skill_Average` | mean(Technical_Skill_Score, Communication_Score) | Combined soft+hard skill signal |
| `Project_Experience` | Projects + Internships | Combined "practical experience" measure |

All three use only information known **before** a placement decision, so they
don't introduce data leakage.

In [ ]:
df_fe = engineer_features(df_clean)
df_fe[["Internships", "Previous_Internship", "Total_Experience",
       "Technical_Skill_Score", "Communication_Score", "Skill_Average",
       "Projects", "Project_Experience"]].head()


## 7. Data Leakage Check

**Data leakage** happens when a model has access, directly or indirectly, to
information it wouldn't realistically have at prediction time — making
evaluation look better than it actually is.

Checks performed:
- `Student_ID` is excluded from model inputs (it's an identifier, not a real predictive feature).
- `Placement_Status` (the target) is never included as an input feature.
- All engineered features use only pre-placement information.
- Scaling/encoding will be fit **only** on the training set (via an sklearn `Pipeline`), never on the full dataset before splitting.

## 8. Preprocessing Setup

We separate features (X) from the target (y), then build a `ColumnTransformer`
that scales numeric columns and one-hot encodes categorical columns. This is
wrapped in a `Pipeline` together with each model, so preprocessing is always
fit only on training data — never on test data.

In [ ]:
numeric_features, categorical_features = get_feature_columns()
X = df_fe[numeric_features + categorical_features]
y = (df_fe["Placement_Status"] == "Placed").astype(int)

print("X shape:", X.shape)
print("y distribution:\n", y.value_counts())


## 9. Train-Test Split

We use an 80/20 split with `random_state=42` for reproducibility, and
`stratify=y` so both sets keep a similar Placed/Not-Placed ratio.

**Training data** is what the model learns patterns from.
**Testing data** is held out and used only to check how well those patterns
generalize to students the model has never seen. Training and evaluating on
the same data would give an overly optimistic, misleading result.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Train size:", X_train.shape[0], " Test size:", X_test.shape[0])


## 10. Model Training

**Logistic Regression** — a linear model that estimates the probability of
an outcome as a weighted combination of input features. Simple, fast, and a
good baseline.

**Random Forest** — an ensemble of many decision trees, each trained on a
random subset of data/features. A single tree can overfit to noise, but
averaging across many trees tends to generalize better.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
}

results = {}
fitted_pipelines = {}

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", build_preprocessor()),
        ("classifier", model),
    ])
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
    }
    fitted_pipelines[name] = pipeline

    print(f"=== {name} ===")
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred, target_names=["Not Placed", "Placed"]))


## 11. Model Evaluation & Comparison

We don't rely on accuracy alone. Key terms:
- **True Positive / True Negative:** correct "Placed" / correct "Not Placed" predictions.
- **False Positive:** predicted Placed, but student wasn't (false alarm).
- **False Negative:** predicted Not Placed, but student actually was (missed case).
- **Precision:** of predicted "Placed" students, how many actually were placed.
- **Recall:** of actually placed students, how many did we correctly catch.
- **F1-Score:** balances precision and recall into one number.

In [ ]:
results_df = pd.DataFrame(results).T
results_df


In [ ]:
final_model_name = max(results, key=lambda name: results[name]["F1-Score"])
print("Selected final model (highest F1-Score):", final_model_name)
final_pipeline = fitted_pipelines[final_model_name]


**Interpretation:** Logistic Regression and Random Forest perform almost
identically (F1-Score differs by ~0.004) — with a gap this small, the choice
isn't strongly meaningful; either would be defensible. The modest ~63%
accuracy (vs. a 50% random baseline) is expected: this synthetic dataset was
deliberately generated with a meaningful random-noise component, similar to
how real placement outcomes depend on factors beyond what's measured here
(interview mood, company fit, timing, luck).

## 12. Feature Importance

For the final Random Forest model, we extract which features it relied on
most. Note: importance reflects usefulness for *prediction*, not proof of
real-world *causation*.

In [ ]:
preprocessor = final_pipeline.named_steps["preprocessor"]
clf = final_pipeline.named_steps["classifier"]

num_features = preprocessor.transformers_[0][2]
cat_features = preprocessor.transformers_[1][2]
ohe = preprocessor.named_transformers_["cat"]
cat_names = list(ohe.get_feature_names_out(cat_features))
all_feature_names = list(num_features) + cat_names

if hasattr(clf, "feature_importances_"):
    importances = clf.feature_importances_
    imp_series = pd.Series(importances, index=all_feature_names).sort_values(ascending=False)
    print(imp_series)

    plt.figure(figsize=(8, 6))
    imp_series.head(12)[::-1].plot(kind="barh", color="teal")
    plt.title(f"Feature Importance ({final_model_name})")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()
else:
    coefs = pd.Series(clf.coef_[0], index=all_feature_names).sort_values(ascending=False)
    print(coefs)


## 13. New Student Prediction

We can now use the trained pipeline to predict placement for a brand-new
student, using the same `predict_new_student()` function used by the
Streamlit app (`src/prediction.py`) — so the notebook and the app always
give consistent results.

In [ ]:
from prediction import predict_new_student

example_student = {
    "Age": 21, "CGPA": 8.2, "Attendance_Percentage": 88, "Study_Hours_Per_Day": 4.5,
    "Backlogs": 0, "Internships": 2, "Projects": 3, "Certifications": 2,
    "Communication_Score": 78, "Technical_Skill_Score": 82, "Aptitude_Score": 75,
    "Gender": "Female", "Extracurricular_Activities": "Yes", "Previous_Internship": "Yes",
}

result = predict_new_student(example_student, model_path="../models/model.pkl")
print("Prediction:", result["prediction"])
print(f"Estimated probability of placement: {result['probability'] * 100:.1f}%")


**Note:** this probability comes directly from the trained model — it is a model estimate, not a guarantee of any real student's outcome.

## 14. Conclusion

This project walked through a complete ML workflow: understanding and
cleaning synthetic student data, exploring relationships visually, engineering
a few combined features, training and fairly comparing two classification
models, and using the final model to predict outcomes for new students.

**Key takeaways:**
- CGPA was the strongest individual correlate of placement, though even it
  was only moderately correlated (~0.18) — no single factor dominates.
- Logistic Regression and Random Forest performed almost identically here.
- ~63% accuracy is modest but expected, given the deliberate randomness built
  into this synthetic dataset — a reminder that real-world placement outcomes
  depend on more than structured data can capture.
- Because the dataset is synthetic, these specific findings should be read as
  a demonstration of *method*, not as real-world conclusions about student
  placement.

See the project README for full documentation, limitations, and future
improvement ideas.